# Feature Engineering — E-Commerce Fraud Data

**Objective**: Transform raw data into a model-ready feature matrix.

Steps covered:
1. Load cleaned data
2. Geolocation merge
3. Behavioral feature engineering
4. One-hot encoding of categoricals
5. Scaling numerical features
6. SMOTE class balancing (training set only)
7. Save processed datasets

---

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data_loader import load_fraud_data, load_ip_country, load_creditcard
from preprocessing import (
    clean_fraud_data,
    clean_creditcard,
    merge_ip_to_country,
    engineer_fraud_features,
    prepare_fraud_data_for_modeling,
    prepare_creditcard_for_modeling,
    split_and_resample,
)
from sklearn.preprocessing import StandardScaler

print('✅ Imports successful')

## Pipeline 1: E-Commerce Fraud Data

In [ ]:
# Step 1: Load raw data
fraud_raw = load_fraud_data('../data/raw/Fraud_Data.csv')
ip_country = load_ip_country('../data/raw/IpAddress_to_Country.csv')

# Step 2: Clean
fraud_clean = clean_fraud_data(fraud_raw)

# Step 3: Geolocation
fraud_geo = merge_ip_to_country(fraud_clean, ip_country)

# Step 4: Feature Engineering
fraud_featured = engineer_fraud_features(fraud_geo)

print(f'\nFinal shape after feature engineering: {fraud_featured.shape}')
fraud_featured[['time_since_signup', 'hour_of_day', 'day_of_week',
                 'transaction_count_per_user', 'transaction_velocity']].describe()

In [ ]:
# Step 5: Prepare for modeling (encode + extract X, y)
X_fraud, y_fraud = prepare_fraud_data_for_modeling(fraud_featured)

print(f'Feature matrix: {X_fraud.shape}')
print(f'\nFeature list:')
print(X_fraud.columns.tolist())

In [ ]:
# Step 6: Stratified split + SMOTE (SMOTE applied ONLY to training set)
print('='*55)
print('FRAUD DATA — Train/Test Split + SMOTE')
print('='*55)

X_fraud_train, X_fraud_test, y_fraud_train, y_fraud_test = split_and_resample(
    X_fraud, y_fraud, test_size=0.2, random_state=42, use_smote=True
)

print(f'\n⚠️  SMOTE applied ONLY to training set — test set untouched.')
print(f'Test class distribution:\n{y_fraud_test.value_counts()}')

In [ ]:
# Apply StandardScaler — fit ONLY on training data
scaler_fraud = StandardScaler()

# Identify numerical columns to scale
num_cols_fraud = X_fraud.select_dtypes(include=[np.number]).columns.tolist()

X_fraud_train_scaled = X_fraud_train.copy()
X_fraud_test_scaled = X_fraud_test.copy()

X_fraud_train_scaled[num_cols_fraud] = scaler_fraud.fit_transform(
    X_fraud_train[num_cols_fraud]
)
X_fraud_test_scaled[num_cols_fraud] = scaler_fraud.transform(
    X_fraud_test[num_cols_fraud]
)

print('✅ StandardScaler fit on training set, applied to both train and test.')
print(f'   Train shape: {X_fraud_train_scaled.shape}')
print(f'   Test shape:  {X_fraud_test_scaled.shape}')

In [ ]:
# Save processed data
os.makedirs('../data/processed', exist_ok=True)

pd.concat([X_fraud_train_scaled, pd.Series(y_fraud_train, name='class').reset_index(drop=True)], axis=1)\
  .to_csv('../data/processed/fraud_train.csv', index=False)

pd.concat([X_fraud_test_scaled, pd.Series(y_fraud_test, name='class').reset_index(drop=True)], axis=1)\
  .to_csv('../data/processed/fraud_test.csv', index=False)

import joblib
joblib.dump(scaler_fraud, '../models/scaler_fraud.pkl')

print('✅ Saved: fraud_train.csv, fraud_test.csv, scaler_fraud.pkl')

## Pipeline 2: Credit Card Data

In [ ]:
# Load and clean
cc_raw = load_creditcard('../data/raw/creditcard.csv')
cc_clean = clean_creditcard(cc_raw)

# Prepare (scales Amount + Time, separates X/y)
X_cc, y_cc, scaler_cc = prepare_creditcard_for_modeling(cc_clean)

print(f'Feature matrix: {X_cc.shape}')

In [ ]:
print('='*55)
print('CREDIT CARD — Train/Test Split + SMOTE')
print('='*55)

X_cc_train, X_cc_test, y_cc_train, y_cc_test = split_and_resample(
    X_cc, y_cc, test_size=0.2, random_state=42, use_smote=True
)

print(f'\n⚠️  SMOTE applied ONLY to training set — test set untouched.')
print(f'Test class distribution:\n{y_cc_test.value_counts()}')

In [ ]:
# Save credit card processed data
pd.concat([pd.DataFrame(X_cc_train, columns=X_cc.columns),
           pd.Series(y_cc_train, name='Class').reset_index(drop=True)], axis=1)\
  .to_csv('../data/processed/creditcard_train.csv', index=False)

pd.concat([pd.DataFrame(X_cc_test, columns=X_cc.columns),
           pd.Series(y_cc_test, name='Class').reset_index(drop=True)], axis=1)\
  .to_csv('../data/processed/creditcard_test.csv', index=False)

joblib.dump(scaler_cc, '../models/scaler_creditcard.pkl')

print('✅ Saved: creditcard_train.csv, creditcard_test.csv, scaler_creditcard.pkl')

## Summary: Resampling Justification

### Why SMOTE over Random Undersampling?

| Approach | Pros | Cons |
|----------|------|------|
| **SMOTE** (chosen) | Preserves all data; generates synthetic minority samples via k-NN interpolation | May create noisy samples near class boundary |
| Random Undersampling | Fast; removes noise | Discards potentially informative majority samples |
| Class weights | No data modification | Less effective on extreme imbalance |

**Reference**: Chawla et al. (2002) — [SMOTE Paper](https://arxiv.org/abs/1106.1813)

### Critical Rule ✅
> SMOTE was applied **only to the training set**. The test set preserves the original class distribution to simulate real-world inference conditions.